# Newswendor - Simulation

In [1]:
# Re-import sim.py whenever it changes on disk, so edits there apply without a kernel restart.
%load_ext autoreload
%autoreload 2

# Data handling: polars is used instead of pandas because the sales/price tables run to
# millions of rows and the reshape + join steps below are much cheaper on its query engine.
import polars as pl
from pathlib import Path

# Modelling: sklearn for scaling/PCA, scipy for the kernel density estimate of WTP.
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import scipy
import heapq
from collections import defaultdict, deque
import cvxpy as cp
from tqdm.auto import tqdm   # notebook widget in Jupyter, text bar elsewhere

# Plotting.
import seaborn as sns
import matplotlib.pyplot as plt

from sim import Demand, Env, Inventory, SupplierInfo, Supplier

## Import data

The three raw M5 CSVs are read straight from `data/` at the repo root. Their schemas are documented
below so the joins further down (`d` for daily grain, `wm_yr_wk` for weekly price grain) are easy to follow.

## M5 Forecasting Dataset Column Definitions

------------------------------
## 📅 calendar.csv
This file links daily sales data to specific dates, financial weeks, calendar events, and social assistance schedules.

* date: The calendar date formatted as YYYY-MM-DD.
* wm_yr_wk: A unique ID indicating the Walmart financial week. It is used to join this file with sell_prices.csv.
* weekday: The name of the day (e.g., Saturday, Sunday).
* wday: A numerical ID representing the day of the week, starting with Saturday (1) through Friday (7).
* month: The numerical month of the year (1–12).
* year: The numerical year.
* d: The specific day identifier used as headers in the training sets (e.g., d_1, d_2 ... up to d_1969).
* event_name_1: The name of a special holiday or cultural event happening on that day (e.g., SuperBowl, Thanksgiving, EidAlFitr). It is blank if no event occurs.
* event_type_1: The category classification of event_name_1 (e.g., Sporting, Cultural, National, Religious).
* event_name_2: Used on rare days where two distinct public events overlap on the same date.
* event_type_2: The category classification of event_name_2.
* snap_CA, snap_TX, snap_WI: Binary flags (0 or 1) indicating whether the stores in California, Texas, or Wisconsin allow SNAP (food stamp) purchases on that specific day.

------------------------------
## 📈 sales_train_validation.csv / sales_train_evaluation.csv
These horizontal datasets contain the historical unit transaction columns that establish the product hierarchy.

* id: A concatenated unique text key representing the specific product identifier combined with the store ID (e.g., HOBBIES_1_001_CA_1_validation).
* item_id: The granular code assigned to the product item itself (e.g., HOBBIES_1_001).
* dept_id: The department code separating broad categories (e.g., HOBBIES_1, FOODS_3).
* cat_id: The top-level category label (Hobbies, Foods, or Household).
* store_id: The store identifier where the product was sold (e.g., CA_1, TX_3, WI_2).
* state_id: The broader geographical US region (CA, TX, or WI).
* d_1 to d_1941: Sequential column headers tracking the exact quantity of units sold on that corresponding day index. This maps directly to the d column in calendar.csv.

------------------------------
## 🏷️ sell_prices.csv
This table allows you to extract revenue and price elasticity metrics across different stores.

* store_id: The store location code matching the sales matrix.
* item_id: The specific product identifier matching the sales matrix.
* wm_yr_wk: The target Walmart financial week index matching calendar.csv.
* sell_price: The specific retail dollar price of the given item at that store location during that operational week. Prices are tracked weekly because retail prices typically stay fixed for 7-day spans.






In [2]:
data_folder = Path.cwd().parents[1] / 'data'

# --- ingestion: no data is read here; scan_csv only inspects the header ---
calendar_lf = pl.scan_csv(data_folder / 'calendar.csv')
prices_lf   = pl.scan_csv(data_folder / 'sell_prices.csv')
sales_lf    = pl.scan_csv(data_folder / 'sales_train_evaluation.csv')

In [3]:
STORE = ['CA_1', 'CA_2']

# collect_schema() reads the header only, it does not execute the plan.
day_cols  = [c for c in sales_lf.collect_schema().names() if c.startswith('d_')]
day_order = sorted(day_cols, key=lambda c: int(c.removeprefix('d_')))   # chronological, not lexical

# --- demand: wide daily matrix -> long (item_id, d, demand) + financial week ---
demand_lf = (
    sales_lf
    .filter(pl.col('store_id').is_in(STORE))          # predicate pushed into the scan
    .select(['item_id', *day_cols])
    .unpivot(index='item_id', on=day_cols, variable_name='d', value_name='demand')
    .join(calendar_lf.select('d', 'wm_yr_wk'), on='d', how='inner')
)

# --- prices: same store, join keys + price only (projection pushdown drops the rest) ---
prices_store_lf = (
    prices_lf
    .filter(pl.col('store_id').is_in(STORE))
    .select('item_id', 'wm_yr_wk', 'sell_price')
) 

# --- item-day fact table ---
merged_lf = (
    demand_lf
    .join(prices_store_lf, on=['item_id', 'wm_yr_wk'], how='inner')   # inner: drops unpriced item-weeks
    .with_columns(
        pl.col('item_id').str.split('_').list.slice(0, 2).list.join('_').alias('dept_id'),
        pl.col('d').str.split('_').list.get(-1).cast(pl.Int32).alias('day_n'),
    )
)

# --- department x day aggregate ---
dept_day_lf = (
    merged_lf
    .group_by('dept_id', 'day_n')
    .agg(
        pl.col('demand').sum().alias('total_demand'),
        (pl.col('demand') * pl.col('sell_price')).sum().alias('total_weighted_price'),
    )
    .with_columns(
        (pl.col('total_weighted_price') / pl.col('total_demand')).round(3).alias('price_avg')
    )
    .filter(pl.col('price_avg').is_not_nan())     # zero-demand department-days give 0/0
)

wide_df = (
    dept_day_lf
    .sort('day_n')                                # column order = order of first appearance
    .collect()
    .pivot(index='dept_id', on='day_n', values='total_demand')
)

# Pivot on the integer day, then relabel — the columns are already chronological,
# so this rename maps each column onto its own data.
wide_df.columns = ['dept_id'] + [f'd_{c}' for c in wide_df.columns[1:]]

day_spine = pl.LazyFrame(
    {'day_n': [int(c.removeprefix('d_')) for c in day_order]},
    schema={'day_n': pl.Int32},
)

wide_lf = (
    dept_day_lf.select('dept_id').unique()
    .join(day_spine, how='cross')                                     # every dept x every day
    .join(dept_day_lf.select('dept_id', 'day_n', 'total_demand'),
          on=['dept_id', 'day_n'], how='left')
    .with_columns(pl.col('total_demand').fill_null(0))
    .group_by('dept_id')
    .agg(pl.col('total_demand').sort_by('day_n'))                     # explicit ordering
    .with_columns(pl.col('total_demand').list.to_struct(fields=day_order))
    .unnest('total_demand')
    .sort(by='dept_id')
)

wide_df = wide_lf.collect()          # .collect(engine="streaming") if memory is tight


## Demand

### Demand  willingness to pay

In [4]:
# --- demand side: wide daily matrix -> long (item_id, d, demand) ---
demand_ca_1 = sales_lf.lazy().filter(pl.col('store_id').is_in(STORE))

# Every column named d_* is a day of history; the rest are hierarchy/metadata columns.
# collect_schema() reads the header only, doesn't execute the plan.
day_cols = [col for col in sales_lf.collect_schema().names() if col.startswith('d_')]

demand_ca_1 = demand_ca_1.select(['item_id'] + day_cols).unpivot(
    index='item_id',
    on=day_cols,
    variable_name='d',
    value_name='demand'
)

# Attach the Walmart financial week for each day so the weekly prices can be joined on.
demand_ca_1 = demand_ca_1.join(
    calendar_lf.lazy().select(pl.col('d', 'wm_yr_wk')),
    on='d',
    how='inner'
)

# --- price side: same store, keep only the join keys and the price itself ---
prices_ca_1 = prices_lf.lazy().filter(
    pl.col('store_id').is_in(STORE)
).select(
    pl.col('item_id', 'wm_yr_wk', 'sell_price')
)

# Inner join drops item-days with no active price (item not stocked that week) — see note above.
merge = demand_ca_1.join(prices_ca_1, on=['item_id', 'wm_yr_wk'], how='inner')

# Recover the department from the item code: HOBBIES_1_001 -> HOBBIES_1 (first two underscore tokens).
merge = merge.with_columns(
    pl.col("item_id").str.split("_").list.slice(0, 2).list.join("_").alias("dept_id")
)

# Roll the item-level table up to one row per (day, department).
ca_1_lf = merge.with_columns(
    (pl.col('demand') * pl.col('sell_price')).alias('total_price')
).group_by(
    ['d', 'dept_id']
).agg(
    pl.col('demand').sum().alias('total_demand'),
    pl.col('total_price').sum().alias('total_weighted_price')
).with_columns(
    (pl.col('total_weighted_price') / pl.col('total_demand')).round(3).alias('price_avg'),
    pl.col('d').str.split("_").list.get(-1).cast(pl.Int32).alias('day_n')
).drop_nans(subset='price_avg')

# Nothing executes until collect() is called — this triggers predicate/projection pushdown
# (filters and column selection happen as early as possible in the scan).
ca_1 = ca_1_lf.collect()
print(ca_1.shape)
ca_1

(13557, 6)


d,dept_id,total_demand,total_weighted_price,price_avg,day_n
str,str,i64,f64,f64,i32
"""d_1029""","""FOODS_2""",747,2487.26,3.33,1029
"""d_1347""","""HOBBIES_1""",1538,5196.98,3.379,1347
"""d_1758""","""FOODS_1""",2086,5466.88,2.621,1758
"""d_1878""","""HOBBIES_2""",154,371.48,2.412,1878
"""d_1364""","""HOUSEHOLD_2""",680,3262.14,4.797,1364
…,…,…,…,…,…
"""d_1784""","""FOODS_1""",1986,5215.19,2.626,1784
"""d_1785""","""FOODS_3""",6066,16070.6,2.649,1785
"""d_1788""","""HOUSEHOLD_2""",1026,4748.64,4.628,1788


In [5]:
wtp_kde = {
    df['dept_id'][0]: scipy.stats.gaussian_kde(df['price_avg'], weights=df['total_demand'])
    for df in ca_1.partition_by('dept_id')
}

# 1-wtp_kde['HOUSEHOLD_1'].integrate_box_1d(-np.inf, 3.5)

In [6]:
ca_1.group_by('dept_id').agg(pl.col('price_avg').min().alias('min'), pl.col('price_avg').max().alias('max'), 
                             pl.col('price_avg').mean().alias('mean').round(1)).sort('dept_id')

dept_id,min,max,mean
str,f64,f64,f64
"""FOODS_1""",1.968,3.333,2.5
"""FOODS_2""",2.435,3.992,3.3
"""FOODS_3""",1.437,2.701,2.3
"""HOBBIES_1""",1.587,5.912,3.7
"""HOBBIES_2""",0.664,3.973,2.3
"""HOUSEHOLD_1""",2.96,4.704,3.9
"""HOUSEHOLD_2""",4.395,5.988,5.2


### Demand module

In [7]:
demand = Demand(wide_df, wtp_kde)

## Enviroment

In [8]:
SEED = 42   # single seed for the whole run: demand, quotes, lead times, forecasts
sim = Env(seed=SEED)

## Inventory

In [9]:
initial_order = {
    'FOODS_1': {
        'quantity': 270,
        'price': 2.5,   
        'shelf_life': 60,
    },
    'FOODS_2': {
        'quantity': 5000,
        'price': 3.3,   
        'shelf_life': 120,
    },
    'FOODS_3': {
        'quantity': 5000,
        'price': 2.3,   
        'shelf_life': 10,
    },
    'HOBBIES_1': {
        'quantity': 5000,
        'price': 3.7,   
        'shelf_life': 1000,
    },
    'HOBBIES_2': {
        'quantity': 5000,
        'price': 2.3,   
        'shelf_life': 1000,
    },
    'HOUSEHOLD_1': {
        'quantity': 5000,
        'price': 3.9,   
        'shelf_life': 1000,
    },
    'HOUSEHOLD_2': {
        'quantity': 5000,
        'price': 5.2,   
        'shelf_life': 1000,
    },
}

In [10]:
inventory = Inventory(sim, demand.wtp)

## Supplier

In [11]:
supplier_book = Supplier()

DEPT_DATA = {
    'FOODS_3':     dict(price_mean=2.33, price_std=0.16, lead_min=1, lead_max=4,
                         low =dict(q=[65, 215, 640], margin=0.35),
                         prem=dict(q=[60, 200, 610], margin=0.20)),
    'FOODS_1':     dict(price_mean=2.30, price_std=0.20, lead_min=5, lead_max=10,
                         low =dict(q=[25, 85, 250], margin=0.45),
                         prem=dict(q=[25, 80, 230], margin=0.30)),
    'FOODS_2':     dict(price_mean=3.31, price_std=0.27, lead_min=3, lead_max=7,
                         low =dict(q=[17, 60, 170], margin=0.30),
                         prem=dict(q=[17, 55, 170], margin=0.20)),
    'HOBBIES_1':   dict(price_mean=3.63, price_std=0.76, lead_min=5, lead_max=10,
                         low =dict(q=[30, 100, 300], margin=0.40),
                         prem=dict(q=[20, 75, 250], margin=0.30)),
    'HOBBIES_2':   dict(price_mean=2.19, price_std=0.41, lead_min=7, lead_max=14,
                         low =dict(q=[20, 100, 200], margin=0.50),
                         prem=dict(q=[20, 100, 200], margin=0.35)),
    'HOUSEHOLD_1': dict(price_mean=3.87, price_std=0.27, lead_min=7, lead_max=14,
                         low =dict(q=[25, 85, 250], margin=0.38),
                         prem=dict(q=[25, 75, 230], margin=0.28)),
    'HOUSEHOLD_2': dict(price_mean=5.18, price_std=0.23, lead_min=5, lead_max=10,
                         low =dict(q=[50, 170, 500], margin=0.30),
                         prem=dict(q=[50, 160, 480], margin=0.20)),
}

TIER_MULTIPLIERS = [1.0, 0.5, 0.2]  # p1, p2, p3 relative to lot price at min order qty


def build_price_dists(price_mean, price_std, margin):
    cv = price_std / price_mean  # constant coefficient of variation across tiers
    lot_price = price_mean * (1 - margin)
    return [
        scipy.stats.norm(loc=lot_price * m, scale=lot_price * m * cv)
        for m in TIER_MULTIPLIERS
    ]


for dept, cfg in DEPT_DATA.items():
    lead_time_low = scipy.stats.expon(loc=cfg['lead_max'], scale=cfg['lead_max'])
    lead_time_prem = scipy.stats.expon(loc=cfg['lead_min'], scale=cfg['lead_min'])

    supplier_book.add_supplier(
        SupplierInfo(
            name=f'{dept}_low',
            product=dept,
            price=build_price_dists(cfg['price_mean'], cfg['price_std'], cfg['low']['margin']),
            lead_time=lead_time_low,
            order_quantity=cfg['low']['q'],
        ),
        SupplierInfo(
            name=f'{dept}_premium',
            product=dept,
            price=build_price_dists(cfg['price_mean'], cfg['price_std'], cfg['prem']['margin']),
            lead_time=lead_time_prem,
            order_quantity=cfg['prem']['q'],
        ),
    )

In [12]:
quotes = supplier_book.order_proposal('HOUSEHOLD_1', rng=sim.supplier_rng)

## Simulation

In [21]:
EPOCH = 100
RUN_DAYS = 1900

PRODUCT = demand.product

shelf_life_dict = {
    'FOODS_1'       : 270,
    'FOODS_2'       : 120,
    'FOODS_3'       : 12,
    'HOBBIES_1'     : 1000,
    'HOBBIES_2'     : 1000,
    'HOUSEHOLD_1'   : 400,
    'HOUSEHOLD_2'   : 400,
}
wastage = []

In [22]:
FORECAST_WINDOW = 30

# Per-product forecast accuracy, in demand.product order. Frozen normals rather than fixed
# numbers: forecast quality is itself uncertain, so every call draws a fresh MAPE and the
# long-run average per department is its loc. Keep scale well under loc — draws are floored
# at 0, so a wide scale piles probability mass on that floor (i.e. perfect forecasts).
FORECAST_MAPE_BY_DEPT = {
    'FOODS_1':     scipy.stats.norm(loc=0.20, scale=0.03),
    'FOODS_2':     scipy.stats.norm(loc=0.15, scale=0.03),
    'FOODS_3':     scipy.stats.norm(loc=0.10, scale=0.03),
    'HOBBIES_1':   scipy.stats.norm(loc=0.15, scale=0.03),
    'HOBBIES_2':   scipy.stats.norm(loc=0.25, scale=0.03),
    'HOUSEHOLD_1': scipy.stats.norm(loc=0.10, scale=0.03),
    'HOUSEHOLD_2': scipy.stats.norm(loc=0.15, scale=0.03),
}


def _draw_mape(value, rng):
    """A frozen scipy distribution becomes a draw; a plain number passes through.

    Floored at 0 because a MAPE cannot be negative, and a normal always can dip below it.
    """
    if hasattr(value, 'rvs'):
        return max(0.0, float(value.rvs(random_state=rng)))

    return float(value)


def forecast(env, data, product, mape=FORECAST_MAPE_BY_DEPT, window=FORECAST_WINDOW,
             seed=None, products=None, history=False):
    """Forecast `product`'s daily demand for each of the next `window` days.

    `data` : (days, products) actual demand, i.e. generate_demand().T
    `product` : which column of `data` to forecast.
    `mape` : a number, a frozen scipy distribution, or a dict of either keyed by product
             (the default). Distributions are drawn fresh on every call, so accuracy varies
             day to day rather than being pinned.
    `products` : column order of `data`; defaults to demand.product.
    `seed` : int or np.random.Generator for the forecast errors; defaults to env.forecast_rng,
             so forecasts follow the simulation's seed.
    `history` : if truthy, also return all actual demand strictly before env.clock,
                oldest first (today is not included); if falsy, that part is empty.

    Returns (past, forecast): the forecast has one value per forecast day (fewer near
    the end of the demand path). The
    per-day relative errors are rescaled so their mean over the window — the MAPE — lands
    exactly on the drawn target. Days with zero actual demand are left out of that mean,
    since a percentage error is undefined there, and are forecast as zero.
    """
    rng = np.random.default_rng(env.forecast_rng if seed is None else seed)
    data = np.asarray(data)
    n_days = data.shape[0]

    products = list(products if products is not None else demand.product)
    col = products.index(product)

    # Forecast starts tomorrow, truncated by the end of the demand path. min() rather than
    # clamping both ends to n_days-1, which collapsed to an empty slice on the last days
    # and silently returned a zero forecast.
    start = min(env.clock + 1, n_days)
    end = min(start + window, n_days)

    y_true = data[start:end, col].astype(float)

    if isinstance(mape, dict):
        mape = mape[product]
    target = _draw_mape(mape, rng)

    # Shape of the magnitudes is arbitrary — the rescale below is what fixes the MAPE.
    # Only days with nonzero actuals count toward MAPE, so rescale over those alone.
    scored = y_true > 0
    magnitudes = rng.uniform(0.1, 1.0, size=y_true.size)
    if scored.any():
        magnitudes = magnitudes * (target / magnitudes[scored].mean())

    signs = rng.choice([-1.0, 1.0], size=y_true.size)
    y_pred = y_true * (1 + signs * magnitudes)

    # Demand cannot go negative; only reachable if a day's magnitude exceeds 1.0, i.e. a
    # target MAPE above roughly 0.55. Clipping there would pull the MAPE off target.
    past = data[:env.clock, col] if history else data[:0, col]

    return past, np.round(y_pred)


In [23]:
# Process event
def process_events(time, event):

    if event[0] == 'stockup':
        _, product, quantity, price, supplier_name, ordered_on = event
        inventory.stockup({product: {'quantity': quantity, 'shelf_life': shelf_life_dict[product]}})
        policy.record_arrival(product, quantity, lead_time=time - ordered_on)
        return None

    if event[0] == 'expire':
        flushout = inventory.flushout_expired(event[1])
        wastage.append((time, flushout))
        return flushout     # [products, units written off], for the daily log

## Policy: newswendor, order only from 1 supplier

In [24]:
LEAD_WINDOW = 5   # realised lead times kept per product, pooled over its suppliers


class Policy:

    def __init__(self, inventory, z=1.65):
        self.z = z   # safety factor for the reorder point (1.65 ~ 95% of lead-time cycles)
        self.reorder_point = {}
        for product in demand.product:
            self.reorder_point[product] = 0

        self.on_order = defaultdict(int)                             # product -> units in transit
        self.lead_obs = defaultdict(lambda: deque(maxlen=LEAD_WINDOW))  # product -> last lead times

        self.markup = {
                        'FOODS_1': 0.4,
                        'FOODS_2': 0.25,
                        'FOODS_3': 0.3,
                        'HOBBIES_1': 0.35,
                        'HOBBIES_2': 0.4,
                        'HOUSEHOLD_1': 0.32,
                        'HOUSEHOLD_2': 0.25}

        # Decision horizon (days an order should cover) per product. Non-perishables are
        # capped at 100 days rather than their 1000-day shelf life, so a newsvendor order
        # covers a sensible cycle instead of years of demand. FOODS_1 is 60 to match its
        # opening lot's shelf life in initial_order, so the opening stock can sell before
        # it expires.
        self.shelf_life = {
                        'FOODS_1': 60,
                        'FOODS_2': 120,
                        'FOODS_3': 10,
                        'HOBBIES_1': 100,
                        'HOBBIES_2': 100,
                        'HOUSEHOLD_1': 100,
                        'HOUSEHOLD_2': 100}

    def reset_epoch(self):
        """Clear stock in transit; the calendar is wiped on sim.reset(). Lead-time history is kept."""
        self.on_order.clear()

    def record_order(self, placed):
        for o in placed:
            self.on_order[o['product']] += o['quantity']

    def record_arrival(self, product, quantity, lead_time):
        self.on_order[product] -= quantity
        self.lead_obs[product].append(lead_time)

    def lead_stats(self, product):
        """Mean and std of the product's lead time over its last LEAD_WINDOW deliveries,
        whichever supplier shipped them.

        Until the window is full, falls back to an equal mix of the product's suppliers'
        lead-time distributions: mean of the means, and the mixture std, which adds the
        spread between suppliers to each one's own variance.
        """
        obs = self.lead_obs[product]
        if len(obs) == LEAD_WINDOW:
            return np.mean(obs), np.std(obs, ddof=1)

        dists = [s.lead_time for s in supplier_book.by_product[product]]
        means = np.array([dist.mean() for dist in dists])
        variances = np.array([dist.var() for dist in dists])
        return means.mean(), np.sqrt(variances.mean() + means.var())

    def update_reorder_point(self, product):
        """d*E[L] + z*sigma_LT, with demand and the product's lead-time uncertainty."""
        _, fc = forecast(sim, demand_data, product, window=self.shelf_life[product])
        d, sigma = np.mean(fc), np.std(fc)

        L, sL = self.lead_stats(product)
        sigma_lt = np.sqrt(L * sigma**2 + d**2 * sL**2)   # demand noise + lead-time noise
        self.reorder_point[product] = d * L + self.z * sigma_lt

    def newswendor(self, product, quotes):
        product_idx = demand.product.index(product)
        decision_period = self.shelf_life[product]

        
        quote_price = []
        for quote in quotes:
            quote_price += list(quote.price)

        max_cost = np.max(quote_price)
        max_price = max_cost*(1+self.markup[product]) 
        historical_data, forecast_data = forecast(sim, demand_data, product, window=decision_period, history=True)


        cost_u, cost_o = (max_price-max_cost), max_cost
        critical_ratio = (cost_u)/(cost_u+cost_o)

        # Non-overlapping shelf-life windows; drop the partial window at the end so reshape works.
        n_windows = len(historical_data) // decision_period
        if n_windows == 0:
            # Not even one full window of history yet: fall back to the forecast total.
            return np.round(np.sum(forecast_data))

        historical_data = np.asarray(historical_data[:n_windows * decision_period])
        window_demand = historical_data.reshape(-1, decision_period).sum(axis=1)
        D = np.round(np.percentile(window_demand, critical_ratio*100))

        return D

    def order_lp(self, product, quotes, order_qty):
        """Pick one supplier and how many lots of each size to buy from it.

        Returns (quote, lots) where lots[t] is the number of lots of quote.order_quantity[t],
        or None if the solve fails. order_purchase takes one lot per call, so place each lot
        separately.
        """
        decision_period = self.shelf_life[product]
        _, forecast_data = forecast(sim, demand_data, product, window=decision_period)

        d, sigma = np.mean(forecast_data), max(np.std(forecast_data), 1e-6) # demand rate, variability

        cost = np.array([q.price for q in quotes], dtype=float)          # (suppliers, tiers) unit price
        lot  = np.array([q.order_quantity for q in quotes], dtype=float)  # (suppliers, tiers) lot size
        # Each supplier's own lead-time distribution, so the choice still reflects which one
        # is faster (observed lead times are pooled by product and can't tell them apart).
        # Not the quote's lead_time: that is the exponential's loc, i.e. its minimum.
        lead_mean = np.array([q.supplier.lead_time.mean() for q in quotes], dtype=float)
        lead_std  = np.array([q.supplier.lead_time.std() for q in quotes], dtype=float)

        margin = np.max(cost)*(self.markup[product])  # profit lost per unit short
        cost_o = np.max(cost)                         # cost of a unit left over (as in newswendor)

        # Expected units lost before each supplier's delivery lands (normal loss function).
        # Lead-time demand variance = demand noise over L days + lead-time noise at rate d.
        d_l = d*lead_mean
        sigma_l = np.sqrt(lead_mean*sigma**2 + d**2*lead_std**2)
        position = inventory.inventory_status().get(product, {'quantity': 0})['quantity'] # quantity in stock

        k = (position-d_l)/sigma_l
        shortfall = sigma_l * (scipy.stats.norm.pdf(k) - k*scipy.stats.norm.sf(k))   # (suppliers,)

        x = cp.Variable(lot.shape, integer=True, nonneg=True)   # lots per (supplier, tier)
        y = cp.Variable(len(quotes), boolean=True)              # 1 for the chosen supplier
        under = cp.Variable(integer=True, nonneg=True)              # units short of order_qty
        over  = cp.Variable(integer=True, nonneg=True)              # units above order_qty

        qty = cp.sum(cp.multiply(lot, x))             # total units ordered

        constraints = [
            cp.sum(y) == 1,                                          # one supplier
            x <= 100 * cp.reshape(y, (len(quotes), 1), order='C'),   # lots only from the chosen supplier
            qty + under - over == order_qty,          # gap to the newsvendor quantity
        ]
        obj = (cp.sum(cp.multiply(cost * lot, x))     # purchase cost
               + margin * (shortfall @ y)             # lost sales while waiting for delivery
               + margin * under                       # lot too small for the cycle
               + cost_o * over)                       # lot too big, risk of waste
        prob = cp.Problem(cp.Minimize(obj), constraints)
        prob.solve(solver=cp.HIGHS)

        if prob.status != cp.OPTIMAL:
            return None
        s = int(np.argmax(y.value))
        return quotes[s], np.round(x.value[s]).astype(int)

    def decision(self, inventory, **karg):
        """Reorder any product at or below its reorder point; returns the orders placed today."""
        placed_today = []
        status = inventory.inventory_status()
        for product in self.reorder_point:
            # inventory_status only lists products currently in stock
            on_hand = status.get(product, {'quantity': 0})['quantity']
            position = on_hand + self.on_order[product]     # count stock in transit

            self.update_reorder_point(product)              # demand and lead times move daily
            if position <= self.reorder_point[product]:
                quotes = supplier_book.order_proposal(product, rng=sim.supplier_rng)
                order_qty = self.newswendor(product, quotes)

                result = self.order_lp(product, quotes, order_qty)
                if result is None:
                    continue
                quote, lots = result    # lots[t] = number of lots of quote.order_quantity[t]

                # Every lot size in one order: a single delivery, one lead-time draw.
                placed = supplier_book.order_purchase(
                    sim,
                    {quote.supplier: {'lots': [int(n) for n in lots], 'price': quote.price}},
                    demand_index=0,
                )
                self.record_order(placed)
                placed_today.extend(placed)

        return placed_today


In [27]:
policy = Policy(inventory)

# One seed per epoch: epoch k always replays the same demand, quotes, lead times and
# forecasts, so any single epoch can be reproduced with sim.reset(seed=epoch_seeds[k]).
epoch_seeds = np.arange(EPOCH)

# Daily log, one row per (epoch, day, product). Column lists rather than a row per dict:
# 100 epochs x 366 days x 7 products is ~256k rows, built into a DataFrame once at the end.
log = defaultdict(list)

for epoch in tqdm(range(EPOCH), desc='epochs'):
    sim.reset(seed=int(epoch_seeds[epoch]))
    inventory.reset(keep_price=False)
    policy.reset_epoch()

    # Demand first: the opening newsvendor quantities below forecast from it.
    demand_data = demand.generate_demand(seed=sim.demand_rng).T.astype(int)
    demand_gen = iter(demand_data)

    # Opening stock = each product's newsvendor quantity on day 0, from today's quotes.
    # Prices and opening shelf lives still come from initial_order. After sim.reset() so
    # the opening lots' 'expire' events land on the fresh calendar.
    opening_order = {
        product: {
            'quantity': int(policy.newswendor(product, supplier_book.order_proposal(product, rng=sim.supplier_rng))),
            'price': initial_order[product]['price'],
            'shelf_life': initial_order[product]['shelf_life'],
        }
        for product in demand.product
    }
    inventory.stockup(opening_order)
    inventory.set_price(opening_order)

    

    while sim.clock <= RUN_DAYS:
        todays_demand = next(demand_gen)
        day_wastage = defaultdict(int)
        if sim.clock in sim.event_calendar:
            for event in sim.event_calendar[sim.clock]:
                flushout = process_events(sim.clock, event) # process event
                if event[0] == 'expire':
                    for product, units in zip(*flushout):
                        day_wastage[product] += units
            sim.remove_event(sim.clock)

        # Sell today's demand after deliveries and expiries, before the policy reorders.
        # sell_stock applies each product's price via its WTP curve; unpriced products sell 0.
        sold = inventory.sell_stock(
            {product: int(todays_demand[i]) for i, product in enumerate(demand.product)}
        )

        forecast_data = {p: forecast(sim, demand_data, p)[1] for p in demand.product}
           
                    
        # Orders placed today, summed per product (quantity and purchase cost).
        day_order_qty, day_order_cost = defaultdict(int), defaultdict(float)
        for order in policy.decision(inventory):
            day_order_qty[order['product']] += order['quantity']
            day_order_cost[order['product']] += order['cost']

        # End-of-day snapshot. inventory_status only lists products in stock, hence .get.
        status = inventory.inventory_status()
        for i, product in enumerate(demand.product):
            log['epoch'].append(epoch)
            log['seed'].append(int(epoch_seeds[epoch]))
            log['day'].append(sim.clock)
            log['product'].append(product)
            log['demand'].append(int(todays_demand[i]))
            log['sold'].append(sold.get(product, 0))
            log['price'].append(float(status.get(product, {}).get('price', np.nan)))
            log['wastage'].append(day_wastage[product])
            # Purchase details only on the day an order is placed; empty otherwise.
            ordered = product in day_order_qty
            log['order_qty'].append(day_order_qty[product] if ordered else None)
            log['order_cost'].append(float(day_order_cost[product]) if ordered else None)
            log['stock'].append(status.get(product, {}).get('quantity', 0))

        sim.clock_step()

sim_log = (
    pl.DataFrame(log, schema={
        'epoch': pl.Int32, 'seed': pl.Int64, 'day': pl.Int32, 'product': pl.Utf8,
        'demand': pl.Int64, 'sold': pl.Int64, 'price': pl.Float64, 'wastage': pl.Int64,
        'order_qty': pl.Int64, 'order_cost': pl.Float64, 'stock': pl.Int64,
    })
    .with_columns(pl.col('price').fill_nan(None))   # unpriced -> null, like the other gaps
)

results_folder = Path.cwd().parents[1] / 'results'   # repo root, alongside code/ and data/
results_folder.mkdir(exist_ok=True)
# Parquet keeps the column types, has no row limit, and is far smaller and faster than
# Excel. Read back with pl.read_parquet(...).
sim_log.write_parquet(results_folder / 'Newswendor_SingleSupplier_ConstantPrice.parquet')
sim_log

epochs:   0%|          | 0/100 [00:00<?, ?it/s]

epoch,seed,day,product,demand,sold,price,wastage,order_qty,order_cost,stock
i32,i64,i32,str,i64,i64,f64,i64,i64,f64,i64
0,0,0,"""FOODS_1""",1305,703,2.5,0,null,null,69918
0,0,0,"""FOODS_2""",1073,602,3.3,0,null,null,95974
0,0,0,"""FOODS_3""",7602,3590,2.3,0,62080,19244.8,55779
0,0,0,"""HOBBIES_1""",1798,894,3.7,0,null,null,147799
0,0,0,"""HOBBIES_2""",36,14,2.3,0,null,null,4959
…,…,…,…,…,…,…,…,…,…,…
99,99,1900,"""FOODS_3""",5779,2729,2.3,0,null,null,51845
99,99,1900,"""HOBBIES_1""",1437,715,3.7,0,null,null,59905
99,99,1900,"""HOBBIES_2""",142,56,2.3,0,null,null,6542
